In [ ]:
from helper_func import *
import helper_func as hf
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())

In [ ]:
train = pd.read_csv('data/train.csv')

In [ ]:
train.head()

In [ ]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = hf.clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

In [ ]:
train.head()

In [ ]:
train['misspelling_count'] = train['clean_text'].apply(count_misspellings)

In [ ]:
train

In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from textstat import flesch_reading_ease, gunning_fog
from nltk.sentiment import SentimentIntensityAnalyzer
from collections import Counter
import numpy as np

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('vader_lexicon')


def count_special_characters(text):
    """
    Counts the number of special characters in the given text.

    :param text: The text to analyze for special characters.
    :return: Count of special characters.
    """
    # Regex to find non-alphanumeric and non-space characters
    special_chars = re.findall(r"[^A-Za-z0-9\s]", text)
    return len(special_chars)

def add_text_features(df, text_column,status='Pre'):
    """
    Enhances the DataFrame with multiple text-based features, including the count of special characters.

    :param df: DataFrame containing the essay texts.
    :param text_column: Column name containing text data.
    :return: DataFrame with added features.
    """
    # Tokenization, Sentence Splitting, and POS Tagging
    df[f'{status}_tokens'] = df[text_column].apply(nltk.word_tokenize)
    df[f'{status}_sentences'] = df[text_column].apply(nltk.sent_tokenize)
    df[f'{status}_pos_tags'] = df[f'{status}_tokens'].apply(nltk.pos_tag)

    # Basic counts
    df[f'{status}_word_count'] = df[f'{status}_tokens'].apply(len)
    df[f'{status}_sentence_count'] = df[f'{status}_sentences'].apply(len)
    df[f'{status}_avg_sentence_length'] = df[f'{statfile:///home/laptop/Pictures/Screenshots/Screenshot%20from%202024-04-14%2012-18-24.pngus}_word_count'] / df[f'{status}_sentence_count']

    # Vocabulary richness
    df[f'{status}_lexical_diversity'] = df[f'{status}_tokens'].apply(lambda x: len(set(x)) / len(x) if x else 0)

    # Readability scores
    df[f'{status}_flesch_reading_ease'] = df[text_column].apply(flesch_reading_ease)
    df[f'{status}_gunning_fog_index'] = df[text_column].apply(gunning_fog)

    # Sentiment analysis
    sia = SentimentIntensityAnalyzer()

    df[f'{status}_sentiment_score'] = df[text_column].apply(lambda x: sia.polarity_scores(x)['compound'])

    # Advanced vocabulary usage
    
    english_stopwords = set(stopwords.words('english'))

    df[f'{status}_advanced_vocab_usage'] = df[f'{status}_tokens'].apply(lambda x: len([word for word in x if word.lower() not in english_stopwords and len(word) > 6]))

    # Grammatical errors (Placeholder for actual grammar check logic)
    df[f'{status}_grammar_errors'] = np.random.randint(0, 3, size=len(df))  # Random errors count as a placeholder

    # Special characters count
    df[f'{status}_special_characters_count'] = df[text_column].apply(count_special_characters)

    return df

In [ ]:
# Adding text features
train = add_text_features(train, 'full_text', status= 'Pre')

In [ ]:
train

In [ ]:
text_col = 'clean_text'   # 'segmented_text'


glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)

In [ ]:
misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")

In [ ]:
# Example usage
corrected_words, uncorrected_words = main(misspellings)

In [ ]:
# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))


In [ ]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

In [ ]:
# Apply the parallelization

train = parallelize_dataframe(train, apply_segmentation)

train.head()

In [ ]:
text_col = 'segmented_text'  


glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)


misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")


# Example usage
corrected_words, uncorrected_words = main(misspellings)

# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))

print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")


In [ ]:
# train = add_text_features(train, 'corrected_text', status='Post')
# train.head()

In [ ]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [ ]:
train_essays, _ = preprocess_data(train_essays)

In [ ]:
validation_essays, _ = preprocess_data(validation_essays) #, tfidf_vectorizer=tfidf_vectorizer)

In [ ]:
train_essays.to_parquet('data/clean_data/train_essays.parquet')
validation_essays.to_parquet('data/clean_data/validation_essays.parquet')

In [ ]:
train_essays.columns

In [ ]:
train_essays = pd.read_parquet('data/clean_data/train_essays.parquet')
validation_essays = pd.read_parquet('data/clean_data/validation_essays.parquet')

In [ ]:
import pandas as pd
from sklearn.utils import resample

def sample_df(df, col='score', sub=1000, random_state=None):
    """
    Balances the classes in a DataFrame by resampling.
    
    Parameters:
    - df: DataFrame to be resampled.
    - col: The column name in df that contains class labels.
    - random_state: The random state for reproducibility.
    
    Returns:
    - balanced_df: A DataFrame with balanced classes.
    """
    class_counts = df[col].value_counts()
    target_count = max(int(class_counts.median()) - sub, class_counts.min())  # Ensure target_count is positive
    
    balanced_df = pd.DataFrame()

    for class_label in df[col].unique():
        class_subset = df[df[col] == class_label]
        
        if len(class_subset) > target_count:
            # Downsample majority classes
            class_subset = resample(class_subset,
                                    replace=False,
                                    n_samples=target_count,
                                    random_state=random_state)
        else:
            # Upsample minority classes
            class_subset = resample(class_subset,
                                    replace=True,
                                    n_samples=target_count,
                                    random_state=random_state)
        
        balanced_df = pd.concat([balanced_df, class_subset], axis=0)
    
    # Shuffle the DataFrame to mix the classes well
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return balanced_df


# Balance only the training DataFrame
train_essays = sample_df(train_essays, col='score', random_state=42)

validation_essays = sample_df(validation_essays, col='score', sub=0, random_state=42)

In [ ]:
print(train_essays['score'].value_counts())
print(validation_essays['score'].value_counts())

In [ ]:
STATUS = 'Post'

if STATUS == 'Post':
       
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences'
       , 'Pre_pos_tags','corrected_text', 'segmented_text',
       'Post_tokens', 'Post_sentences', 'Post_pos_tags']

else:
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences', 'Pre_pos_tags',
        'corrected_text', 'segmented_text',]



train_df = train_essays.copy()
val_df = validation_essays.copy()

train_df.drop(columns=drop_cols, inplace= True)
val_df.drop(columns=drop_cols, inplace= True)



In [ ]:
train_df.head()

In [ ]:
feature_cols = []

for col in train_df.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [ ]:
from sklearn.preprocessing import StandardScaler

train_labels = train_df['score']

val_labels = val_df['score']

train_features = train_df[feature_cols]

val_features = val_df[feature_cols]

scaler = StandardScaler()

train_feats_scaled = scaler.fit_transform(train_features)

val_feats_scaled = scaler.transform(val_features)


import pickle

with open('data/scalers/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
import tensorflow as tf

# Assuming train_labels and val_labels are your label arrays with classes 1-6

train_labels_one_hot = tf.keras.utils.to_categorical(train_labels - 1, num_classes=6)
val_labels_one_hot = tf.keras.utils.to_categorical(val_labels - 1, num_classes=6)


In [ ]:
import tensorflow as tf

train_class = tf.data.Dataset.from_tensor_slices((train_feats_scaled, 
                                                train_labels_one_hot)).shuffle(len(train_features)).batch(32)

val_class = tf.data.Dataset.from_tensor_slices((val_feats_scaled, 
                                              val_labels_one_hot)).batch(32)


In [ ]:
# from sklearn.utils import class_weight

# class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
# class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}

# class_weights

In [ ]:
val_labels.shape

In [ ]:
import tensorflow as tf
import numpy as np

def quadratic_weighted_kappa(y_true, y_pred):
    """
    Calculates the Quadratic Weighted Kappa (QWK) aka Cohen's Kappa with quadratic weights.
    
    Parameters:
    y_true (tensor): The ground truth labels as one-hot encoded vectors.
    y_pred (tensor): The predicted probability distributions.
    
    Returns:
    float: The QWK score.
    """
    # Convert predictions to class indices
    y_pred_classes = tf.argmax(y_pred, axis=-1)
    y_pred_one_hot = tf.one_hot(y_pred_classes, depth=y_true.shape[-1])

    # Calculate the confusion matrix
    confusion_matrix = tf.math.confusion_matrix(tf.argmax(y_true, axis=1), y_pred_classes, num_classes=y_true.shape[-1])

    # Normalize the confusion matrix
    confusion_matrix = tf.cast(confusion_matrix, dtype=tf.float32)
    confusion_matrix = confusion_matrix / tf.reduce_sum(confusion_matrix)

    # Calculate weights
    num_classes = tf.shape(confusion_matrix)[0]
    weights = tf.cast(tf.range(num_classes), dtype=tf.float32) / tf.cast(num_classes - 1, dtype=tf.float32)
    weights = tf.square(tf.expand_dims(weights, -1) - tf.expand_dims(weights, 0))

    # Calculate the observed and expected agreement
    observed = tf.reduce_sum(weights * confusion_matrix)
    expected = tf.reduce_sum(
        weights * tf.matmul(
            tf.reshape(tf.reduce_sum(confusion_matrix, axis=1), [-1, 1]),
            tf.reshape(tf.reduce_sum(confusion_matrix, axis=0), [1, -1])
        )
    )

    # Calculate the kappa score
    kappa = 1.0 - observed / expected
    return kappa


In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import schedules, AdamW, RMSprop

def build_classification_model(hp, input_shape=(train_feats_scaled.shape[1],)):
    
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)  # number of hidden layers
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)  # neurons in each hidden layer
    l2_reg = hp.Float("l2_reg", min_value=1e-6, max_value=1e-2, sampling="log")  # L2 regularization
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")  # learning rate

    optimizer_choice = hp.Choice("optimizer_choice", ['adam', 'sgd', 'RMSprop', 'Adagrad', 'Adadelta', 'Nadam', 'Ftrl', 'L-BFGS'])

    # Learning rate schedulers
    lr_schedule = schedules.ExponentialDecay(initial_learning_rate=learning_rate,
                                             decay_steps=10000,
                                             decay_rate=0.9)

    if optimizer_choice == 'adam':
        optimizer = AdamW(learning_rate=lr_schedule)
    elif optimizer_choice == 'sgd':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
    elif optimizer_choice == 'RMSprop':
        optimizer = RMSprop(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adagrad':
        optimizer = tf.keras.optimizers.Adagrad(learning_rate=lr_schedule)
    elif optimizer_choice == 'Adadelta':
        optimizer = tf.keras.optimizers.Adadelta(learning_rate=lr_schedule)
    elif optimizer_choice == 'Nadam':
        optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)
    else:
        optimizer = tf.keras.optimizers.Ftrl(learning_rate=lr_schedule)

    model = tf.keras.models.Sequential()

    # Input layer
    model.add(tf.keras.Input(shape=input_shape))  # Explicit input layer

    model.add(tf.keras.layers.Dense(n_neurons, 
                                    activation='relu', 
                                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))

    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.05)  # dropout rate

    # Hidden layers with Batch Normalization and Dropout
    for _ in range(n_hidden):
        model.add(tf.keras.layers.BatchNormalization())
        
        model.add(tf.keras.layers.Dense(n_neurons, 
                                        activation='relu', 
                                        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        
        model.add(tf.keras.layers.Dropout(rate=dropout_rate))

    # Output layer for 6 classes
    model.add(tf.keras.layers.Dense(6, activation='softmax'))
    
    
    
#     from tensorflow.keras.metrics import AUC

#     # For binary classification
#     roc_auc = AUC(curve='ROC')

#     # For multi-class classification (assuming one-vs-all approach)
#     roc_auc_multi = AUC(curve='ROC', multi_label=True, num_labels=6)




    # Then, you can add it to your model's compile method as a metric
    model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=[ 
                       tf.keras.metrics.CategoricalAccuracy(),
                       tf.keras.metrics.F1Score( average='weighted'),
                       tf.keras.metrics.AUC(curve='ROC', multi_label=True, num_labels=6),
                       quadratic_weighted_kappa])


    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)

    return model



from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import keras_tuner as kt

model_checkpoint = ModelCheckpoint('data/models/best_standard_model_epoch.keras', 
                                   save_best_only=True, monitor='val_loss', mode='min')

early_stopping = EarlyStopping(monitor='val_loss', patience=5, 
                               restore_best_weights=True)

call_backs = [model_checkpoint, early_stopping]

tuner = kt.BayesianOptimization(
    build_classification_model,
    objective='val_loss',
    max_trials=8,
    num_initial_points= 2,
    seed=1,
    overwrite=True,
    directory='data/models/tuner_2',
    project_name='standard',
)

tuner.search(train_class, epochs=1000, 
             validation_data= val_class, callbacks = call_backs, verbose=2)      # class_weight=class_weights_dict,

best_standard_model = tuner.get_best_models(num_models=1)[0]


best_standard_model.save('data/models/best_standard_model.keras')

In [ ]:
best_standard_model.summary()

In [ ]:
best_standard_model.fit(train_class, epochs=1000, class_weight=class_weights_dict)

In [ ]:
# Evaluate the model on the validation dataset

results = best_standard_model.evaluate(val_class, verbose=0)

# Print all results

print(results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming val_reg is your validation dataset
# Extract true labels and convert from one-hot encoding to integer labels if necessary
y_true = np.concatenate([y for x, y in val_class], axis=0)

if y_true.ndim > 1 and y_true.shape[1] > 1:  # Check if y_true is one-hot encoded
    y_true = np.argmax(y_true, axis=1)

# Predict the classes with the best model on the validation data

y_pred = best_standard_model.predict(val_feats_scaled)

y_pred_classes = np.argmax(y_pred, axis=1)  # Convert from probabilities to class labels

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plotting the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
print(sorted(val_labels.unique()))
print(sorted(pd.Series(y_pred_classes+1).unique()))

In [ ]:
from sklearn.metrics import cohen_kappa_score

preds = best_standard_model.predict(val_feats_scaled)
preds = np.argmax(preds, axis=1)  # Convert from probabilities to class labels

actuals = val_labels 
predictions = y_pred_classes + 1 

# Calculate the Quadratic Weighted Kappa
qwk_score = cohen_kappa_score(actuals, predictions, weights='quadratic')

print("Quadratic Weighted Kappa score:", qwk_score)
